# Synthetic Eval Statistical Diagnostics

This notebook is for diagnosing what is wrong or healthy in a completed `synthetic_eval.cli` run.

It focuses on four questions:

- is the ensemble mean accurate enough by RMSE/CRPS;
- is there systematic bias, especially low bias in high truth values;
- is uncertainty calibrated by rank histograms, coverage and spread-skill;
- do saved tensor archives agree with the CSV/NPZ diagnostics.

The statistical scores here are diagnostics, not formal independent-pixel hypothesis tests. Grid cells are spatially correlated, and swath repeats for the same day are not independent days.

In [ ]:
from __future__ import annotations

import csv
import json
import os
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
from IPython.display import Markdown, display

try:
    import pandas as pd
except Exception:
    pd = None

DATA_ROOT = Path(os.environ.get("DATA_ROOT", "/mnt/sciml/a.sadreev/sea_ice_data"))
OUT_DIR = Path(
    os.environ.get(
        "SYNTH_EVAL_OUT",
        DATA_ROOT / "synthetic_eval_swath_30days_x32_r4_saved",
    )
)

if not OUT_DIR.exists():
    candidates = sorted(DATA_ROOT.glob("synthetic_eval*"), key=lambda p: p.stat().st_mtime, reverse=True)
    if not candidates:
        raise FileNotFoundError(f"No synthetic_eval output directories found under {DATA_ROOT}")
    print(f"Configured OUT_DIR does not exist: {OUT_DIR}")
    OUT_DIR = candidates[0]
    print(f"Using latest synthetic_eval directory instead: {OUT_DIR}")

ARRAYS_DIR = OUT_DIR / "arrays"
SAMPLES_DIR = OUT_DIR / "samples"

# Tensor diagnostics can be expensive. Increase these after the notebook runs once.
MAX_ARCHIVES = int(os.environ.get("SYNTH_EVAL_MAX_ARCHIVES", "240"))
PIXEL_STRIDE = int(os.environ.get("SYNTH_EVAL_PIXEL_STRIDE", "8"))
BOOTSTRAP_REPS = int(os.environ.get("SYNTH_EVAL_BOOTSTRAP_REPS", "1000"))
RNG = np.random.default_rng(12345)

print("OUT_DIR      =", OUT_DIR)
print("ARRAYS_DIR   =", ARRAYS_DIR)
print("SAMPLES_DIR  =", SAMPLES_DIR)
print("MAX_ARCHIVES =", MAX_ARCHIVES)
print("PIXEL_STRIDE =", PIXEL_STRIDE)

In [ ]:
def read_json(path: Path, default=None):
    if not path.exists():
        return default
    with open(path) as f:
        return json.load(f)


def read_csv_table(path: Path):
    if not path.exists():
        return None
    if pd is not None:
        return pd.read_csv(path)
    with open(path, newline="") as f:
        return list(csv.DictReader(f))


def table_rows(table):
    if table is None:
        return []
    if pd is not None and hasattr(table, "to_dict"):
        return table.to_dict("records")
    return table


def display_table(rows, n=30):
    if pd is not None:
        display(pd.DataFrame(rows).head(n))
    else:
        display(rows[:n])


def fnum(value, default=np.nan):
    try:
        return float(value)
    except Exception:
        return default


def bootstrap_ci(values, reps=BOOTSTRAP_REPS, q=(2.5, 50.0, 97.5)):
    arr = np.asarray(values, dtype=np.float64)
    arr = arr[np.isfinite(arr)]
    if arr.size == 0:
        return (np.nan, np.nan, np.nan)
    if arr.size == 1:
        return (float(arr[0]), float(arr[0]), float(arr[0]))
    idx = RNG.integers(0, arr.size, size=(reps, arr.size))
    means = arr[idx].mean(axis=1)
    return tuple(float(x) for x in np.percentile(means, q))


def load_valid_mask(metadata, shape):
    mask_path = metadata.get("mask_path") if metadata else None
    if not mask_path or not Path(mask_path).exists():
        return np.ones(shape, dtype=bool)
    valid = np.load(mask_path)
    if valid.ndim == 3:
        valid = valid[0]
    if valid.shape != shape:
        print(f"valid mask shape mismatch: {valid.shape} vs {shape}; using all pixels")
        return np.ones(shape, dtype=bool)
    return valid.astype(bool)


def evaluation_mask(metadata, observed_mask, valid_mask):
    region = (metadata or {}).get("eval_region", "all")
    obs = observed_mask.astype(bool)
    if region == "observed":
        return valid_mask & obs
    if region == "unobserved":
        return valid_mask & ~obs
    return valid_mask


def rank_counts(ensemble, truth, mask, seed=0):
    ens = np.asarray(ensemble, dtype=np.float64)
    y = np.asarray(truth, dtype=np.float64)
    less = np.sum(ens < y[None, ...], axis=0)
    ties = np.sum(ens == y[None, ...], axis=0)
    rng = np.random.default_rng(seed)
    ranks = less + rng.integers(0, ties + 1)
    return np.bincount(ranks[mask].reshape(-1).astype(np.int64), minlength=ens.shape[0] + 1)


def central_coverage(ensemble, truth, level, mask):
    lo_q = (1.0 - level) / 2.0
    hi_q = 1.0 - lo_q
    lo = np.quantile(ensemble, lo_q, axis=0)
    hi = np.quantile(ensemble, hi_q, axis=0)
    hit = (truth >= lo) & (truth <= hi)
    return float(hit[mask].mean()) if mask.any() else np.nan


def verdict_ratio(value, lo, hi):
    if not np.isfinite(value):
        return "missing"
    if lo <= value <= hi:
        return "ok"
    return "bad"


## 1. Run Inputs And Files

In [ ]:
metadata = read_json(OUT_DIR / "metadata.json", default={})
agg = read_csv_table(OUT_DIR / "aggregate_metrics.csv")
per_case = read_csv_table(OUT_DIR / "per_case_metrics.csv")
spread_bins = read_csv_table(OUT_DIR / "spread_skill_bins.csv")
agg_rows = table_rows(agg)
per_rows = table_rows(per_case)
spread_rows = table_rows(spread_bins)
tensor_paths = sorted(SAMPLES_DIR.glob("**/*.npz"))

display(Markdown("### Metadata"))
display(metadata)
print("aggregate rows      =", len(agg_rows))
print("per-case rows       =", len(per_rows))
print("spread-skill bins   =", len(spread_rows))
print("saved tensor files  =", len(tensor_paths))
print("rank npz exists     =", (ARRAYS_DIR / "rank_histograms.npz").exists())

if tensor_paths:
    for path in tensor_paths[:8]:
        print(path.relative_to(OUT_DIR))


## 2. Aggregate Quality Verdict

This summarizes the CSV metrics. `spread_skill_ratio = spread / RMSE`, so values below 1 usually mean underdispersion. Coverage below nominal also means undercoverage.

In [ ]:
display_table(agg_rows, n=30)

verdict_rows = []
for row in agg_rows:
    variable = row.get("variable")
    mask_type = row.get("mask_type")
    density = fnum(row.get("density"))
    rmse = fnum(row.get("rmse"))
    crps = fnum(row.get("crps"))
    ratio = fnum(row.get("spread_skill_ratio"))
    cov50 = fnum(row.get("coverage_0.5"))
    cov80 = fnum(row.get("coverage_0.8"))
    cov90 = fnum(row.get("coverage_0.9"))
    cov95 = fnum(row.get("coverage_0.95"))
    verdict_rows.append({
        "variable": variable,
        "mask_type": mask_type,
        "density": density,
        "eval_region": row.get("eval_region"),
        "rmse": rmse,
        "crps": crps,
        "spread_skill_ratio": ratio,
        "ratio_verdict": verdict_ratio(ratio, 0.85, 1.15),
        "coverage_50_gap": cov50 - 0.50,
        "coverage_80_gap": cov80 - 0.80,
        "coverage_90_gap": cov90 - 0.90,
        "coverage_95_gap": cov95 - 0.95,
        "coverage_verdict": "ok" if np.nanmin([cov50 - 0.50, cov80 - 0.80, cov90 - 0.90, cov95 - 0.95]) > -0.05 else "undercoverage",
    })

display_table(verdict_rows, n=50)

if verdict_rows:
    labels = [f"{r['variable']} d={r['density']:g}" for r in verdict_rows]
    ratio = np.array([r["spread_skill_ratio"] for r in verdict_rows], dtype=float)
    gaps90 = np.array([r["coverage_90_gap"] for r in verdict_rows], dtype=float)
    fig, axes = plt.subplots(1, 2, figsize=(13, 4))
    axes[0].bar(labels, ratio)
    axes[0].axhline(1.0, color="black", linestyle="--", linewidth=1)
    axes[0].set_title("Spread / RMSE")
    axes[0].tick_params(axis="x", rotation=45)
    axes[1].bar(labels, gaps90)
    axes[1].axhline(0.0, color="black", linestyle="--", linewidth=1)
    axes[1].set_title("90% coverage gap")
    axes[1].tick_params(axis="x", rotation=45)
    plt.tight_layout()
    plt.show()


## 3. Rank Histogram Shape Diagnostics

Flat is calibrated. U-shape means underdispersion. Center-heavy means overdispersion. Right edge larger than left edge means truth is often above all ensemble members, so the ensemble is biased low. The chi-square-like score is a shape score, not a formal p-value.

In [ ]:
rank_path = ARRAYS_DIR / "rank_histograms.npz"
rank_summary = []
if rank_path.exists():
    rank_data = np.load(rank_path)
    keys = sorted(rank_data.files)
    fig, axes = plt.subplots(len(keys), 1, figsize=(10, max(3.0, 2.4 * len(keys))), squeeze=False)
    for ax, key in zip(axes.ravel(), keys):
        counts = rank_data[key].astype(float)
        total = counts.sum()
        probs = counts / total if total else counts
        flat = 1.0 / len(counts)
        ranks = np.arange(len(counts))
        ax.bar(ranks, probs, color="black")
        ax.axhline(flat, color="tab:red", linestyle="--", linewidth=1)
        ax.set_title(key)
        ax.set_ylabel("probability")
        ax.grid(True, axis="y", alpha=0.25)

        edge_ratio = (probs[0] + probs[-1]) / (2 * flat)
        center_ratio = probs[len(probs)//3:2*len(probs)//3].mean() / flat
        right_left = (probs[-1] - probs[0]) / flat
        chi2_like = ((counts - total * flat) ** 2 / max(total * flat, 1.0)).sum()
        rank_summary.append({
            "key": key,
            "total_ranks": int(total),
            "edge_ratio_vs_flat": float(edge_ratio),
            "center_ratio_vs_flat": float(center_ratio),
            "right_minus_left_edges_flat_units": float(right_left),
            "chi2_like_not_pvalue": float(chi2_like),
            "main_issue": "bias_low + underdispersed" if right_left > 0.5 and edge_ratio > 1.4 else ("underdispersed" if edge_ratio > 1.4 else "mixed/ok"),
        })
    axes.ravel()[-1].set_xlabel("truth rank among ensemble members")
    plt.tight_layout()
    plt.show()
    display_table(rank_summary, n=50)
else:
    print("missing:", rank_path)


## 4. Per-Case Bootstrap CIs From CSV

This bootstraps condition-case rows from `per_case_metrics.csv`. It is useful for stability checks, but it is still optimistic if several rows come from the same physical day.

In [ ]:
ci_rows = []
group_keys = sorted({(r.get("variable"), r.get("mask_type"), r.get("density"), r.get("noise_level")) for r in per_rows})
for key in group_keys:
    sub = [r for r in per_rows if (r.get("variable"), r.get("mask_type"), r.get("density"), r.get("noise_level")) == key]
    for metric, target in [("rmse", None), ("crps", None), ("spread_skill_ratio", 1.0), ("coverage_0.9", 0.9), ("coverage_0.95", 0.95)]:
        vals = [fnum(r.get(metric)) for r in sub]
        lo, med, hi = bootstrap_ci(vals)
        out = {
            "variable": key[0],
            "mask_type": key[1],
            "density": fnum(key[2]),
            "noise_level": fnum(key[3]),
            "metric": metric,
            "n_rows": len(sub),
            "mean_ci_low": lo,
            "mean": med,
            "mean_ci_high": hi,
        }
        if target is not None:
            out["target"] = target
            out["target_inside_ci"] = bool(lo <= target <= hi)
            out["mean_minus_target"] = med - target
        ci_rows.append(out)

display_table(ci_rows, n=80)


## 5. Tensor-Level Bias, Coverage, And Spread Tests

This section uses saved `.npz` archives. It recomputes diagnostics on the exact saved samples and can expose where the CSV aggregates hide a problem. The mask follows `metadata['eval_region']`.

In [ ]:
tensor_rows = []
truth_bin_rows = []
tensor_rank_counts = {}
levels = [0.5, 0.8, 0.9, 0.95]
variables = metadata.get("variables", ["ch0", "ch1"])

selected_paths = tensor_paths[:MAX_ARCHIVES]
print(f"scanning {len(selected_paths)} / {len(tensor_paths)} tensor archives")

for archive_i, path in enumerate(selected_paths):
    z = np.load(path, allow_pickle=False)
    ensemble = z["ensemble"][:, :, ::PIXEL_STRIDE, ::PIXEL_STRIDE]
    truth = z["truth"][:, ::PIXEL_STRIDE, ::PIXEL_STRIDE]
    obs_mask = z["mask"][::PIXEL_STRIDE, ::PIXEL_STRIDE]
    condition_tag = str(z["condition_tag"])
    case_index = int(z["case_index"])
    valid = load_valid_mask(metadata, z["mask"].shape)[::PIXEL_STRIDE, ::PIXEL_STRIDE]
    emask = evaluation_mask(metadata, obs_mask, valid)
    if not emask.any():
        continue
    mean = ensemble.mean(axis=0)
    spread = ensemble.std(axis=0, ddof=1 if ensemble.shape[0] > 1 else 0)

    for ch in range(min(len(variables), truth.shape[0])):
        var = variables[ch]
        err = mean[ch] - truth[ch]
        rmse = float(np.sqrt(np.mean(err[emask] ** 2)))
        bias = float(np.mean(err[emask]))
        mae = float(np.mean(np.abs(err[emask])))
        spr = float(np.sqrt(np.mean(spread[ch][emask] ** 2)))
        ratio = spr / rmse if rmse > 0 else np.nan
        cov = {f"coverage_{level:g}": central_coverage(ensemble[:, ch], truth[ch], level, emask) for level in levels}
        counts = rank_counts(ensemble[:, ch], truth[ch], emask, seed=archive_i + ch)
        tensor_rank_counts[var] = tensor_rank_counts.get(var, np.zeros_like(counts)) + counts
        tensor_rows.append({
            "archive": str(path.relative_to(OUT_DIR)),
            "condition_tag": condition_tag,
            "case_index": case_index,
            "variable": var,
            "n_pixels": int(emask.sum()),
            "bias_mean_minus_truth": bias,
            "rmse": rmse,
            "mae": mae,
            "spread_rms": spr,
            "spread_skill_ratio": ratio,
            **cov,
        })

        tvals = truth[ch][emask]
        eval_err = err[emask]
        q_edges = np.quantile(tvals, [0.0, 0.25, 0.50, 0.75, 1.0])
        q_edges = np.unique(q_edges)
        for lo, hi in zip(q_edges[:-1], q_edges[1:]):
            sel = (tvals >= lo) & (tvals <= hi if hi == q_edges[-1] else tvals < hi)
            if not np.any(sel):
                continue
            truth_bin_rows.append({
                "archive": str(path.relative_to(OUT_DIR)),
                "condition_tag": condition_tag,
                "case_index": case_index,
                "variable": var,
                "truth_bin_lo": float(lo),
                "truth_bin_hi": float(hi),
                "n_pixels": int(sel.sum()),
                "mean_truth": float(tvals[sel].mean()),
                "mean_error": float(eval_err[sel].mean()),
                "rmse": float(np.sqrt(np.mean(eval_err[sel] ** 2))),
            })

print("tensor diagnostic rows =", len(tensor_rows))
display_table(tensor_rows, n=12)


In [ ]:
tensor_ci_rows = []
for var in sorted({r["variable"] for r in tensor_rows}):
    sub = [r for r in tensor_rows if r["variable"] == var]
    for metric, target in [
        ("bias_mean_minus_truth", 0.0),
        ("rmse", None),
        ("mae", None),
        ("spread_skill_ratio", 1.0),
        ("coverage_0.5", 0.5),
        ("coverage_0.8", 0.8),
        ("coverage_0.9", 0.9),
        ("coverage_0.95", 0.95),
    ]:
        vals = [r[metric] for r in sub if metric in r]
        lo, med, hi = bootstrap_ci(vals)
        row = {"variable": var, "metric": metric, "n_archives": len(sub), "ci_low": lo, "mean": med, "ci_high": hi}
        if target is not None:
            row["target"] = target
            row["target_inside_ci"] = bool(lo <= target <= hi)
            row["mean_minus_target"] = med - target
        tensor_ci_rows.append(row)

display_table(tensor_ci_rows, n=80)

if tensor_rows:
    fig, axes = plt.subplots(1, 3, figsize=(15, 4))
    for var in sorted({r["variable"] for r in tensor_rows}):
        sub = [r for r in tensor_rows if r["variable"] == var]
        axes[0].hist([r["bias_mean_minus_truth"] for r in sub], bins=24, alpha=0.55, label=var)
        axes[1].hist([r["spread_skill_ratio"] for r in sub], bins=24, alpha=0.55, label=var)
        axes[2].hist([r["coverage_0.9"] for r in sub], bins=24, alpha=0.55, label=var)
    axes[0].axvline(0.0, color="black", linestyle="--")
    axes[0].set_title("Mean error: ensemble mean - truth")
    axes[1].axvline(1.0, color="black", linestyle="--")
    axes[1].set_title("Spread / RMSE")
    axes[2].axvline(0.9, color="black", linestyle="--")
    axes[2].set_title("90% coverage")
    for ax in axes:
        ax.grid(True, alpha=0.25)
        ax.legend()
    plt.tight_layout()
    plt.show()


## 6. Tensor Rank Histograms And Bias By Truth Range

In [ ]:
if tensor_rank_counts:
    fig, axes = plt.subplots(len(tensor_rank_counts), 1, figsize=(10, 3.2 * len(tensor_rank_counts)), squeeze=False)
    tensor_rank_summary = []
    for ax, (var, counts) in zip(axes.ravel(), sorted(tensor_rank_counts.items())):
        total = counts.sum()
        probs = counts / total if total else counts
        flat = 1.0 / len(counts)
        ax.bar(np.arange(len(counts)), probs, color="black")
        ax.axhline(flat, color="tab:red", linestyle="--", linewidth=1)
        ax.set_title(f"tensor recomputed rank histogram: {var}")
        ax.set_xlabel("truth rank among ensemble members")
        ax.set_ylabel("probability")
        ax.grid(True, axis="y", alpha=0.25)
        tensor_rank_summary.append({
            "variable": var,
            "total_ranks": int(total),
            "edge_ratio_vs_flat": float((probs[0] + probs[-1]) / (2 * flat)),
            "right_minus_left_edges_flat_units": float((probs[-1] - probs[0]) / flat),
        })
    plt.tight_layout()
    plt.show()
    display_table(tensor_rank_summary)
else:
    print("No tensor rank counts. Save tensors with --save-tensors to enable this section.")

if truth_bin_rows:
    summary = []
    for var in sorted({r["variable"] for r in truth_bin_rows}):
        sub = [r for r in truth_bin_rows if r["variable"] == var]
        bins = sorted({(round(r["truth_bin_lo"], 6), round(r["truth_bin_hi"], 6)) for r in sub})
        for lo, hi in bins:
            vals = [r["mean_error"] for r in sub if round(r["truth_bin_lo"], 6) == lo and round(r["truth_bin_hi"], 6) == hi]
            tmean = np.mean([r["mean_truth"] for r in sub if round(r["truth_bin_lo"], 6) == lo and round(r["truth_bin_hi"], 6) == hi])
            ci = bootstrap_ci(vals)
            summary.append({"variable": var, "truth_bin_lo": lo, "truth_bin_hi": hi, "mean_truth": tmean, "error_ci_low": ci[0], "mean_error": ci[1], "error_ci_high": ci[2]})
    display_table(summary, n=80)

    fig, ax = plt.subplots(figsize=(8, 5))
    for var in sorted({r["variable"] for r in summary}):
        sub = [r for r in summary if r["variable"] == var]
        x = [r["mean_truth"] for r in sub]
        y = [r["mean_error"] for r in sub]
        yerr = np.array([[r["mean_error"] - r["error_ci_low"] for r in sub], [r["error_ci_high"] - r["mean_error"] for r in sub]])
        ax.errorbar(x, y, yerr=yerr, marker="o", capsize=3, label=var)
    ax.axhline(0.0, color="black", linestyle="--", linewidth=1)
    ax.set_xlabel("mean truth in bin")
    ax.set_ylabel("ensemble mean - truth")
    ax.set_title("Bias by truth value")
    ax.grid(True, alpha=0.3)
    ax.legend()
    plt.show()
else:
    print("No truth-bin rows available.")


## 7. Interpretation Checklist

- Good RMSE/CRPS with bad rank histograms means the mean reconstruction may be usable, but the posterior uncertainty is not calibrated.
- `spread_skill_ratio < 1` plus rank U-shape means the ensemble is too narrow.
- positive `right_minus_left_edges_flat_units` means truth is often above all samples, so the ensemble is biased low.
- negative `bias_mean_minus_truth` confirms the same low bias directly from saved tensors.
- strong negative bias only in high truth bins means the model smooths or clips high concentration/thickness values.
- undercoverage at 90/95% means confidence intervals are too narrow and should not be trusted as calibrated uncertainty.

If this notebook and the viewer notebook agree, the failure is probably model/sampling/calibration, not a plotting artifact. If they disagree, inspect tensor masks, normalization stats and `eval_region` first.